# Chapter 4: Regression and Prediction

**Author:** Ruben Alfa Richie  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

Regression is the workhorse of predictive modeling:
- Simple and multiple linear regression
- Model assessment and validation
- Factor variables and interactions
- Regression diagnostics
- Polynomial and spline regression
- Generalized additive models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 4.1 Simple Linear Regression

### Theory

**Goal:** Model relationship between response (Y) and single predictor (X)

**Regression Equation:**
$$\hat{y} = b_0 + b_1 x$$

where:
- $b_0$ = intercept (Y when X=0)
- $b_1$ = slope (change in Y per unit X)
- $\hat{y}$ = predicted value

**Least Squares:** Minimizes sum of squared residuals
$$\min \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Formulas:**
$$b_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2} = r \frac{s_y}{s_x}$$

$$b_0 = \bar{y} - b_1 \bar{x}$$

**Key Terms:**
- **Fitted values**: Predictions $\hat{y}_i$
- **Residuals**: Errors $e_i = y_i - \hat{y}_i$
- **RSE (Residual Standard Error)**: Typical residual size
- **R² (R-squared)**: Proportion of variance explained (0-1)

In [ ]:
# Simple linear regression example
np.random.seed(42)
n = 100
X = np.random.uniform(0, 10, n)
y = 5 + 2*X + np.random.normal(0, 2, n)  # True: y = 5 + 2x + noise

# Manual calculation
x_mean, y_mean = X.mean(), y.mean()
b1 = np.sum((X - x_mean) * (y - y_mean)) / np.sum((X - x_mean)**2)
b0 = y_mean - b1 * x_mean

print("SIMPLE LINEAR REGRESSION")
print(f"True model: y = 5 + 2x")
print(f"Estimated: y = {b0:.2f} + {b1:.2f}x")

# Using sklearn
model = LinearRegression()
model.fit(X.reshape(-1, 1), y)
y_pred = model.predict(X.reshape(-1, 1))

# Model quality
residuals = y - y_pred
rse = np.sqrt(np.sum(residuals**2) / (n - 2))
r2 = r2_score(y, y_pred)

print(f"\nModel Quality:")
print(f"R² = {r2:.4f} ({r2*100:.1f}% variance explained)")
print(f"RSE = {rse:.4f} (typical error)")
print(f"Correlation = {np.corrcoef(X, y)[0,1]:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter with regression line
axes[0].scatter(X, y, alpha=0.6, edgecolor='black')
axes[0].plot(X, y_pred, 'r-', linewidth=2, label=f'y = {b0:.2f} + {b1:.2f}x')
axes[0].set_xlabel('X (Predictor)')
axes[0].set_ylabel('y (Response)')
axes[0].set_title(f'Simple Linear Regression (R² = {r2:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
axes[1].scatter(y_pred, residuals, alpha=0.6, edgecolor='black')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot (should be random scatter)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Good model: R² high, residuals randomly scattered around 0")

## 4.2 Multiple Linear Regression

### Theory

**Multiple predictors:**
$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + ... + b_p x_p$$

**Matrix form:**
$$\mathbf{\hat{y}} = \mathbf{X\beta}$$

**Least squares solution:**
$$\mathbf{\hat{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

**Interpretation:**
- $b_j$ = change in Y per unit change in $x_j$, **holding other predictors constant**
- Coefficients adjust for correlations between predictors

**Model Assessment:**
- **Adjusted R²**: Penalizes adding predictors
- **AIC/BIC**: Information criteria (lower = better)
- **F-statistic**: Overall model significance

**Issues:**
- **Multicollinearity**: Predictors highly correlated
- **Overfitting**: Too many predictors for sample size